# Notebook 4: Spatial Neighborhood Analysis

This notebook characterizes the spatial tumor microenvironment using neighborhood enrichment analysis, tumor-centric distance (SKNY) profiling, and ligand-receptor interaction inference.

**Overview:**
1. Squidpy spatial neighborhood enrichment (Delaunay triangulation)
2. SKNY distance analysis: distance from tumor cells across samples
3. Ligand-receptor interaction inference with LIANA (cellphonedb method)
4. Longitudinal tracking of spatial relationships across timepoints

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
from scipy import stats
from statsmodels.stats.multitest import multipletests
import scanpy as sc
import squidpy as sq
import liana as li
import anndata

sc.settings.verbosity = 1
sc.settings.figdir = '../figures/'
plt.rcParams['figure.dpi'] = 150

## 2. Patient Metadata

In [ ]:
# Response group helper
def get_response(pt_num):
    if pt_num <= 8:
        return 'SD'
    elif pt_num <= 16:
        return 'MPR'
    else:
        return 'CR'

# All sample names with Pre/Resection timepoints
sample_name_ls = [
    'Pt-1_Pre',  'Pt-1_Resection',
    'Pt-2_Pre',  'Pt-2_Resection',
    'Pt-3_Pre',  'Pt-3_JustAfter',  'Pt-3_Resection',
    'Pt-4_Pre',  'Pt-4_Resection',
    'Pt-5_Pre',  'Pt-5_JustAfter',  'Pt-5_Resection',
    'Pt-6_Pre',  'Pt-6_Resection',
    'Pt-7_Pre',  'Pt-7_Resection',
    'Pt-8_Pre',  'Pt-8_Resection',
    'Pt-9_Pre',  'Pt-9_Resection',
    'Pt-10_Pre', 'Pt-10_Resection',
    'Pt-11_Pre', 'Pt-11_JustAfter', 'Pt-11_Resection',
    'Pt-12_Pre', 'Pt-12_Resection',
    'Pt-13_Pre', 'Pt-13_JustAfter', 'Pt-13_Resection',
    'Pt-14_Pre', 'Pt-14_Resection',
    'Pt-15_Pre', 'Pt-15_Resection',
    'Pt-16_Pre', 'Pt-16_Resection',
    'Pt-17_Pre', 'Pt-17_Resection',
    'Pt-18_Pre', 'Pt-18_Resection',
    'Pt-19_Pre', 'Pt-19_Resection',
    'Pt-20_Pre', 'Pt-20_Resection',
    'Pt-21_Pre', 'Pt-21_Resection',
    'Pt-22_Pre', 'Pt-22_Resection',
    'Pt-23_Pre', 'Pt-23_Resection',
    'Pt-24_Pre', 'Pt-24_Resection',
]

## 3. Load Data

In [ ]:
# Load the fully annotated concatenated object
adata = sc.read_h5ad('../data/bicrc_integrated_annotated_.h5ad')
print(adata)

# Ensure spatial coordinates are present
if 'spatial' not in adata.obsm:
    raise ValueError('Spatial coordinates not found in adata.obsm["spatial"]. '
                     'Check that the h5ad file includes spatial embeddings.')

## 4. Spatial Neighborhood Enrichment

For each sample, a Delaunay triangulation graph is constructed using squidpy, and neighborhood enrichment scores are computed to identify cell type pairs that co-localize more (or less) than expected by chance.

In [ ]:
def compute_nhood_enrichment(adata_sample, cluster_key='subcluster'):
    """Compute neighborhood enrichment for a single sample.
    
    Returns the nhood_enrichment zscore matrix as a DataFrame.
    """
    sq.gr.spatial_neighbors(adata_sample, coord_type='generic', delaunay=True)
    sq.gr.nhood_enrichment(adata_sample, cluster_key=cluster_key)
    
    zscore_matrix = adata_sample.uns[f'{cluster_key}_nhood_enrichment']['zscore']
    labels = adata_sample.obs[cluster_key].cat.categories
    return pd.DataFrame(zscore_matrix, index=labels, columns=labels)

In [ ]:
# Compute neighborhood enrichment per sample
nhood_results = {}

for sample_name in sample_name_ls:
    if sample_name not in adata.obs['Sample'].values:
        continue
    
    adata_sample = adata[adata.obs['Sample'] == sample_name].copy()
    
    # Ensure subcluster is categorical
    if 'subcluster' in adata_sample.obs.columns:
        adata_sample.obs['subcluster'] = adata_sample.obs['subcluster'].astype('category')
    
    try:
        df_nhood = compute_nhood_enrichment(adata_sample, cluster_key='subcluster')
        pt_num = int(sample_name.split('_')[0].replace('Pt-', ''))
        nhood_results[sample_name] = {
            'zscore': df_nhood,
            'response': get_response(pt_num),
            'timepoint': sample_name.split('_', 1)[1],
        }
        print(f'Processed {sample_name}')
    except Exception as e:
        print(f'Failed {sample_name}: {e}')

print(f'\nCompleted {len(nhood_results)} samples')

In [ ]:
# Visualize neighborhood enrichment heatmap for a representative sample
def plot_nhood_heatmap(df_nhood, title, output_path, vmin=-5, vmax=5):
    """Plot neighborhood enrichment zscore as a heatmap."""
    fig, ax = plt.subplots(figsize=(14, 12))
    sns.heatmap(
        df_nhood, cmap='RdBu_r', center=0, vmin=vmin, vmax=vmax,
        ax=ax, square=True, linewidths=0.3
    )
    ax.set_title(title)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=90, fontsize=7)
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=7)
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.show()

# Plot for a CR and SD representative at Pre-treatment
for sample_name in ['Pt-24_Pre', 'Pt-1_Pre', 'Pt-9_Pre']:
    if sample_name in nhood_results:
        plot_nhood_heatmap(
            nhood_results[sample_name]['zscore'],
            title=f'Neighborhood Enrichment - {sample_name}',
            output_path=f'../figures/04_nhood_{sample_name.replace("-", "").replace("_", "")}.png'
        )

## 5. SKNY Distance Analysis

SKNY (SquidPy Kriging-based Neighborhood distance analYsis) quantifies the distance of each cell from the tumor boundary. Per-sample SKNY h5ad files contain a 'distance_from_tumor' metric in obs.

In [ ]:
# Load per-sample SKNY distance files and aggregate
skny_records = []

for sample_name in sample_name_ls:
    skny_path = f'../data/BICRC_SKNY_tumor_{sample_name}.h5ad'
    try:
        adata_skny = sc.read_h5ad(skny_path)
        df_skny = adata_skny.obs[['subcluster', 'distance_from_tumor']].copy()
        df_skny['Sample'] = sample_name
        pt_num = int(sample_name.split('_')[0].replace('Pt-', ''))
        df_skny['Response'] = get_response(pt_num)
        df_skny['Timepoint'] = sample_name.split('_', 1)[1]
        skny_records.append(df_skny)
        print(f'Loaded SKNY: {sample_name}')
    except FileNotFoundError:
        print(f'SKNY file not found: {sample_name}')

if skny_records:
    df_skny_all = pd.concat(skny_records, ignore_index=True)
    df_skny_all.to_csv('../results/skny_distance_all_samples.csv', index=False)
    print(f'\nAggregated SKNY data: {df_skny_all.shape}')

In [ ]:
# Plot SKNY distance distributions by cell type and response group
if 'df_skny_all' in dir():
    cell_types_of_interest = [
        'c3_POSTN+_CAF', 'c4_MMP11+_CAF', 'c0_KLRG1+_CD8_T', 'c1_Treg',
        'c0_F13A1+_MΦ', 'c7_OLR1+_Mye'
    ]
    df_plot = df_skny_all[
        df_skny_all['subcluster'].isin(cell_types_of_interest) &
        (df_skny_all['Timepoint'] == 'Pre')
    ]

    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    response_colors = {'SD': 'red', 'MPR': 'orange', 'CR': 'blue'}

    for ax, ct in zip(axes.flat, cell_types_of_interest):
        df_ct = df_plot[df_plot['subcluster'] == ct]
        for response, color in response_colors.items():
            vals = df_ct.loc[df_ct['Response'] == response, 'distance_from_tumor']
            if len(vals) > 0:
                sns.kdeplot(vals, ax=ax, color=color, label=response)
        ax.set_title(ct, fontsize=9)
        ax.set_xlabel('Distance from tumor (μm)')
        ax.legend(fontsize=8)

    plt.suptitle('Distance from Tumor by Cell Type and Response (Pre-treatment)', fontsize=12)
    plt.tight_layout()
    plt.savefig('../figures/04_skny_distance_distributions.png', dpi=300, bbox_inches='tight')
    plt.show()

## 6. Ligand-Receptor Interaction Analysis (LIANA)

Cell-cell communication is inferred using LIANA with the CellPhoneDB method and the consensus resource. Analysis focuses on interactions involving tumor-adjacent CAF and myeloid subclusters.

In [ ]:
# Load pre-computed LR results (generated separately due to computational cost)
try:
    df_lr = pd.read_csv('../data/ligand_receptor_around_tumor3.txt', sep='\t')
    print('Loaded pre-computed LR results:', df_lr.shape)
    print(df_lr.columns.tolist())
except FileNotFoundError:
    print('Pre-computed LR file not found. Running LIANA analysis...')
    df_lr = None

In [ ]:
# Run LIANA if not pre-computed
if df_lr is None:
    # Use a subset around tumor cells for LR analysis
    adata_lr = adata[adata.obs['subcluster'].notna()].copy()
    
    lr_results = li.method.cellphonedb(
        adata_lr,
        groupby='subcluster',
        expr_prop=0.1,
        resource_name='consensus',
        inplace=False
    )
    df_lr = lr_results
    df_lr.to_csv('../data/ligand_receptor_around_tumor3.txt', sep='\t', index=False)

In [ ]:
# Filter significant LR interactions
# NR-enriched CAF subtypes: c3_POSTN+_CAF, c4_MMP11+_CAF, c0_Epi_CAF_boundary
# NR-enriched myeloid subtypes: c7_OLR1+_Mye, c5_CAF_Mye_boundary, c3_Epi_Mye_boundary

nr_caf_subclusters = ['c3_POSTN+_CAF', 'c4_MMP11+_CAF', 'c0_Epi_CAF_boundary']
nr_mye_subclusters = ['c7_OLR1+_Mye', 'c5_CAF_Mye_boundary', 'c3_Epi_Mye_boundary']

if df_lr is not None:
    # Filter interactions involving NR-associated cell types
    source_col = 'source' if 'source' in df_lr.columns else df_lr.columns[0]
    target_col = 'target' if 'target' in df_lr.columns else df_lr.columns[1]
    
    df_nr_lr = df_lr[
        (df_lr[source_col].isin(nr_caf_subclusters + nr_mye_subclusters)) |
        (df_lr[target_col].isin(nr_caf_subclusters + nr_mye_subclusters))
    ].copy()
    print(f'NR-associated LR interactions: {len(df_nr_lr)}')
    df_nr_lr.to_csv('../results/lr_nr_enriched_interactions.csv', index=False)

In [ ]:
# Visualize top LR interactions as dot plot
if df_lr is not None and len(df_lr) > 0:
    # Identify score/significance column
    score_col = next(
        (c for c in ['lr_means', 'magnitude_rank', 'specificity_rank', 'magnitude']
         if c in df_lr.columns), None
    )
    pval_col = next(
        (c for c in ['cellphone_pvals', 'pvalue', 'pvals'] if c in df_lr.columns), None
    )
    
    if score_col and pval_col:
        df_sig = df_lr[df_lr[pval_col] < 0.05].nlargest(30, score_col)
        
        fig, ax = plt.subplots(figsize=(10, 8))
        scatter = ax.scatter(
            df_sig[source_col] + ' → ' + df_sig[target_col],
            df_sig['ligand'] + '-' + df_sig['receptor'],
            s=df_sig[score_col] * 20,
            c=-np.log10(df_sig[pval_col] + 1e-10),
            cmap='Reds'
        )
        plt.colorbar(scatter, ax=ax, label='-log10(p-value)')
        ax.set_xticklabels(ax.get_xticklabels(), rotation=90, fontsize=7)
        ax.set_title('Top Ligand-Receptor Interactions (LIANA/CellPhoneDB)')
        plt.tight_layout()
        plt.savefig('../figures/04_liana_lr_dotplot.png', dpi=300, bbox_inches='tight')
        plt.show()

## 7. Longitudinal Neighborhood Dynamics

Tracks changes in spatial co-localization enrichment between key cell type pairs across Pre → JustAfter → Resection timepoints.

In [ ]:
# Extract pairwise co-localization scores for specific cell type pairs of interest
pairs_of_interest = [
    ('c3_POSTN+_CAF', 'Tumor c1'),
    ('c0_KLRG1+_CD8_T', 'Tumor c1'),
    ('c7_OLR1+_Mye', 'c3_POSTN+_CAF'),
]

longitudinal_data = {pair: {} for pair in pairs_of_interest}

for sample_name, result in nhood_results.items():
    df_nhood = result['zscore']
    for pair in pairs_of_interest:
        if pair[0] in df_nhood.index and pair[1] in df_nhood.columns:
            longitudinal_data[pair][sample_name] = df_nhood.loc[pair[0], pair[1]]

print('Longitudinal data collected for', len(longitudinal_data), 'pairs')

In [ ]:
# Plot longitudinal changes in neighborhood enrichment
timepoint_order = ['Pre', 'JustAfter', 'Resection']
response_colors = {'SD': 'red', 'MPR': 'orange', 'CR': 'blue'}

fig, axes = plt.subplots(1, len(pairs_of_interest), figsize=(5 * len(pairs_of_interest), 5))
if len(pairs_of_interest) == 1:
    axes = [axes]

for ax, pair in zip(axes, pairs_of_interest):
    records = []
    for sample_name, score in longitudinal_data[pair].items():
        pt_num = int(sample_name.split('_')[0].replace('Pt-', ''))
        records.append({
            'Sample': sample_name,
            'Patient': f'Pt-{pt_num}',
            'Timepoint': sample_name.split('_', 1)[1],
            'Response': get_response(pt_num),
            'zscore': score,
        })
    df_long = pd.DataFrame(records)
    df_long['Timepoint'] = pd.Categorical(df_long['Timepoint'], categories=timepoint_order, ordered=True)
    df_long = df_long.sort_values(['Patient', 'Timepoint'])

    for patient, grp in df_long.groupby('Patient'):
        response = grp['Response'].iloc[0]
        ax.plot(
            grp['Timepoint'].astype(str), grp['zscore'],
            marker='o', color=response_colors.get(response, 'grey'),
            alpha=0.5, linewidth=1
        )

    ax.axhline(y=0, color='black', linestyle='--', linewidth=0.8)
    ax.set_title(f'{pair[0]}\nvs {pair[1]}', fontsize=9)
    ax.set_xlabel('Timepoint')
    ax.set_ylabel('Neighborhood enrichment (z-score)')

# Add legend
from matplotlib.lines import Line2D
legend_elements = [Line2D([0], [0], color=c, label=r) for r, c in response_colors.items()]
axes[-1].legend(handles=legend_elements, title='Response')

plt.suptitle('Longitudinal Spatial Neighborhood Dynamics', fontsize=12)
plt.tight_layout()
plt.savefig('../figures/04_longitudinal_nhood_dynamics.png', dpi=300, bbox_inches='tight')
plt.show()